# 02buildmap: 3D population map with pydeck (Stage 1)

Klang Valley Reach and Risk, Stage 1, part 2 of 2.

- Input: `data/processed/hex_grid.parquet` (from 01hexgrid), `districts.geojson`, `stations.geojson`, `komuter_stations.geojson`
- Output: `output/stage1_population.html` (open it in your browser; drag to move, right-drag or Ctrl+drag to tilt and rotate)

All visual settings are in Section 0, marked `# <-`.

## Section 0: settings

In [1]:
from pathlib import Path

BASE = Path.cwd()
PROCESSED = BASE / "data" / "processed"
OUTPUT = BASE / "output"
GRID_FILE = PROCESSED / "hex_grid.parquet"
OUT_HTML = OUTPUT / "stage1_population.html"

# ---- Text ----
PAGE_TITLE = "Klang Valley Reach and Risk: Stage 1"                       # <- browser tab
TITLE = "Where the Klang Valley lives"                                     # <- panel heading
SUBTITLE = "People per hexagon (about 0.74 km\u00b2 each). Taller and darker means more people."   # <-
CREDIT = ("Population: Kontur Population 2023-11-01 (CC BY 4.0). Grid: H3 resolution 8. "
          "Basemap: \u00a9 CARTO, \u00a9 OpenStreetMap contributors.")   # <-
NOTE = "Kontur underestimates Putrajaya (about 8,600 vs about 110,000 in the 2020 census)."   # <-

# ---- Basemap and camera ----
MAP_STYLE = "road"          # <- "dark", "light" or "road" (CARTO, no key needed)
VIEW_LAT = 3.0             # <- map centre
VIEW_LON = 101.6           # <-
VIEW_ZOOM = 10.0             # <- higher = closer
VIEW_PITCH = 40             # <- tilt in degrees (0 = flat, 60 = steep)
VIEW_BEARING = -15          # <- rotation in degrees

# ---- Hexagon colours (people per hexagon) ----
BREAKS = [500, 1000, 2500, 5000, 10000]                 # <- class limits
COLOURS = ["#FFF0EA", "#FFCBB7", "#E57A7A", "#9B4174", "#532363", "#1D0E2B"]
SHOW_EMPTY = False          # <- True draws flat grey hexagons where nobody lives
EMPTY_COLOUR = "#3a3a3a"    # <-

# ---- Hexagon shape ----
ELEVATION_SCALE = 0.3       # <- metres of height per person (0.3: 10,000 people = 3 km tall)
COVERAGE = 0.88             # <- hexagon size, 1 = touching, smaller = gaps between them
OPACITY = 0.90              # <-

# ---- Districts and stations ----
DISTRICT_LINE_COLOUR = [60, 60, 60, 140]   # <- RGB + transparency (0 to 255)
SHOW_STATIONS = False       # <- True adds rail stations as small dots (mostly hidden under tall columns)
STATION_COLOUR = [120, 220, 255, 220]        # <-
STATION_RADIUS = 120        # <- metres

# ---- Title panel ----
PANEL_BG = "rgba(17, 17, 20, 0.82)"          # <-
PANEL_TEXT = "#f2f2f2"                       # <-
PANEL_MUTED = "#a8a8b3"                      # <-

In [2]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import pydeck as pdk

grid = pd.read_parquet(GRID_FILE)
districts = gpd.read_file(PROCESSED / "districts.geojson").to_crs(4326)
print("pydeck", pdk.__version__, "|", len(grid), "hexagons |", "{:,} people".format(grid["population"].sum()))

pydeck 0.9.3 | 4003 hexagons | 8,343,652 people


## Step 1: colours and tooltip text for each hexagon

pydeck wants colours as `[red, green, blue]` numbers, so we convert the hex codes from Section 0.

In [3]:
def hex_to_rgb(code):
    code = code.lstrip("#")
    return [int(code[i:i + 2], 16) for i in (0, 2, 4)]

assert len(COLOURS) == len(BREAKS) + 1, "COLOURS needs one more entry than BREAKS"
alpha = int(OPACITY * 255)

data = grid if SHOW_EMPTY else grid[grid["population"] > 0]
data = data.copy()

# VISUAL: colour class for each hexagon
classes = np.searchsorted(BREAKS, data["population"], side="right")
data["color"] = [hex_to_rgb(EMPTY_COLOUR if p == 0 else COLOURS[k]) + [alpha]
                 for p, k in zip(data["population"], classes)]

# Tooltip text
data["place"] = data["district"].str.replace("W.P. ", "", regex=False)
data["pop_txt"] = data["population"].map("{:,}".format)
data["dens_txt"] = data["density"].map("{:,.0f}".format)

data = data[["h3", "population", "color", "place", "pop_txt", "dens_txt"]]
data.head()

,h3,population,color,place,pop_txt,dens_txt
1,8865054593fffff,1,"[255, 240, 234, 229]",Gombak,1,1
2,8865054519fffff,1298,"[229, 122, 122, 229]",Gombak,"1,298","1,510"
4,886505460dfffff,7258,"[83, 35, 99, 229]",Gombak,"7,258","8,448"
5,8865057151fffff,5727,"[83, 35, 99, 229]",Gombak,"5,727","6,659"
6,8865054701fffff,233,"[255, 240, 234, 229]",Gombak,233,271


## Step 2: the layers

- **H3HexagonLayer**: draws a hexagon straight from its H3 code, no polygons needed. `extruded=True` makes it 3D.
- **GeoJsonLayer**: the district outlines.
- **ScatterplotLayer**: rail stations (optional).

In [4]:
# VISUAL: 3D hexagons
hex_layer = pdk.Layer(
    "H3HexagonLayer",
    data,
    get_hexagon="h3",
    get_fill_color="color",
    extruded=True,
    get_elevation="population",
    elevation_scale=ELEVATION_SCALE,
    coverage=COVERAGE,
    pickable=True,
    auto_highlight=True,
)

# VISUAL: district outlines
district_layer = pdk.Layer(
    "GeoJsonLayer",
    json.loads(districts[["district", "geometry"]].to_json()),
    stroked=True,
    filled=False,
    get_line_color=DISTRICT_LINE_COLOUR,
    line_width_min_pixels=1,
)

layers = [district_layer, hex_layer]

# VISUAL: rail stations
if SHOW_STATIONS:
    rr = gpd.read_file(PROCESSED / "stations.geojson")[["lat", "lon"]]
    km = gpd.read_file(PROCESSED / "komuter_stations.geojson")[["lat", "lon"]]
    stations = pd.DataFrame(pd.concat([rr, km], ignore_index=True)).dropna()
    layers.append(pdk.Layer(
        "ScatterplotLayer",
        stations,
        get_position=["lon", "lat"],
        get_fill_color=STATION_COLOUR,
        get_radius=STATION_RADIUS,
        radius_min_pixels=1.5,
        pickable=False,
    ))
    print("Stations drawn:", len(stations))

## Step 3: camera, tooltip and save

In [5]:
view = pdk.ViewState(latitude=VIEW_LAT, longitude=VIEW_LON, zoom=VIEW_ZOOM, pitch=VIEW_PITCH, bearing=VIEW_BEARING)

# VISUAL: tooltip when hovering over a hexagon
tooltip = {
    "html": "<b>{place}</b><br/>{pop_txt} people<br/>{dens_txt} people per km\u00b2",
    "style": {"backgroundColor": "#1b1b22", "color": "#f2f2f2", "fontSize": "13px", "fontFamily": "Segoe UI, Arial, sans-serif", "padding": "8px 10px"},
}

deck = pdk.Deck(layers=layers, initial_view_state=view, map_provider="carto", map_style=MAP_STYLE, tooltip=tooltip)
deck.to_html(str(OUT_HTML), notebook_display=False)
print("Saved", OUT_HTML.name)

Saved stage1_population.html


## Step 4: title panel and legend

pydeck pages have no title or legend, so we add a small HTML panel to the saved file.

In [6]:
# VISUAL: legend rows built from BREAKS and COLOURS
labels = [f"under {BREAKS[0]:,}"]
labels += [f"{a:,} to {b - 1:,}" for a, b in zip(BREAKS[:-1], BREAKS[1:])]
labels += [f"{BREAKS[-1]:,} or more"]
legend_rows = "".join(
    f'<div class="row"><span class="sw" style="background:{c}"></span>{t}</div>'
    for c, t in zip(COLOURS[::-1], labels[::-1])
)

panel = f'''
<style>
  #kr-panel {{ position:absolute; top:12px; left:12px; z-index:10; max-width:min(400px, calc(100vw - 48px));
    background:{PANEL_BG}; color:{PANEL_TEXT}; padding:14px 16px; border-radius:8px;
    font-family:"Segoe UI", Arial, sans-serif; font-size:13px; line-height:1.45; }}
  #kr-panel h1 {{ font-size:24px; margin:0 0 4px; font-weight:600; }}
  #kr-panel .sub {{ color:{PANEL_MUTED}; margin-bottom:10px; }}
  #kr-panel .row {{ display:flex; align-items:center; gap:8px; margin:2px 0; }}
  #kr-panel .sw {{ width:14px; height:14px; border-radius:3px; display:inline-block; }}
  #kr-panel .small {{ color:{PANEL_MUTED}; font-size:11px; margin-top:10px; }}
  @media (max-width: 600px) {{ #kr-panel {{ font-size:12px; padding:10px 12px; }} #kr-panel .small {{ display:none; }} }}
</style>
<div id="kr-panel">
  <h1>{TITLE}</h1>
  <div class="sub">{SUBTITLE}</div>
  {legend_rows}
  <div class="small">{NOTE}</div>
  <div class="small">{CREDIT}</div>
</div>
'''

html = OUT_HTML.read_text(encoding="utf-8")
html = html.replace("<title>pydeck</title>", f"<title>{PAGE_TITLE}</title>", 1)
html = html.replace("</body>", panel + "\n</body>", 1)
OUT_HTML.write_text(html, encoding="utf-8")
print("Saved", OUT_HTML.name, "|", round(OUT_HTML.stat().st_size / 1024), "KB")
print("Open it by double-clicking the file in File Explorer, or run the next cell.")

Saved stage1_population.html | 1014 KB
Open it by double-clicking the file in File Explorer, or run the next cell.


In [7]:
import webbrowser
webbrowser.open(OUT_HTML.resolve().as_uri())

True

___
**END**